# AtlasIR — Day 5 Kaggle compatibility check

Upload this notebook and attach the **private** prepared `atlasir-day05-input.zip` as a Kaggle input (extracted). Enable Internet for pinned uv/Python/dependency setup. Choose the available T4 option for the brief CUDA check. This is a CPU BM25 reproduction plus a separate kernel CUDA probe, not an embedding job or a final timing benchmark.

Run all cells once. Each execution gets a new work directory. Download the two ZIP outputs before stopping the accelerator. Detailed instructions: `docs/DAY05-KAGGLE.md` in the source archive. A separate Google Drive restore is still required before long jobs.


In [ ]:
import hashlib
import json
import os
import subprocess
import sys
import tarfile
import uuid
from pathlib import Path

candidates = list(Path("/kaggle/input").rglob("launcher-input.json"))
assert len(candidates) == 1, "Attach exactly one extracted AtlasIR Day 5 input"
INPUT = candidates[0].parent
manifest = json.loads(candidates[0].read_bytes())
assert manifest["schema_version"] == 1
for name, digest in manifest["sha256"].items():
    path = INPUT / name
    assert path.resolve().is_relative_to(INPUT.resolve())
    assert hashlib.sha256(path.read_bytes()).hexdigest() == digest, name
WORK = Path("/kaggle/working") / ("atlasir-day05-" + uuid.uuid4().hex[:12])
SOURCE = WORK / "source"
SOURCE.mkdir(parents=True)
with tarfile.open(INPUT / "source.tar.gz", "r:gz") as archive:
    seen = set()
    total = 0
    for member in archive:
        relative = Path(member.name)
        total += member.size
        assert member.isfile() and not relative.is_absolute()
        assert ".." not in relative.parts and "\\" not in member.name
        assert member.name not in seen and total <= 128 * 1024 * 1024
        seen.add(member.name)
        destination = SOURCE / relative
        assert destination.resolve().is_relative_to(SOURCE.resolve())
        destination.parent.mkdir(parents=True, exist_ok=True)
        with archive.extractfile(member) as stream:
            destination.write_bytes(stream.read())
print(
    {
        "reference": manifest["reference_run_id"],
        "source_sha256": manifest["sha256"]["source.tar.gz"],
        "work": str(WORK),
    }
)

In [ ]:
# Install only the pinned launcher tool; project packages stay isolated.
TOOLS = WORK / "tools"
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "--target",
        str(TOOLS),
        "uv==" + manifest["uv_version"],
    ],
    check=True,
)
# The uv wheel installs its executable below the target's bin directory.
UV = TOOLS / "bin" / "uv"
assert UV.is_file(), "Pinned uv executable missing from tool target"
ENV = os.environ.copy()
ENV.pop("PYTHONPATH", None)
ENV["UV_CACHE_DIR"] = str(WORK / "uv-cache")
ENV["UV_PROJECT_ENVIRONMENT"] = str(WORK / "project-env")
ENV["UV_PYTHON"] = manifest["python_profile"]
BASE = [
    str(UV),
    "run",
    "--project",
    str(SOURCE),
    "--python",
    manifest["python_profile"],
    "--locked",
    "--no-dev",
]
subprocess.run(BASE + ["python", "-m", "atlasir", "smoke"], env=ENV, cwd=SOURCE, check=True)

In [ ]:
# Probe the existing Kaggle kernel's torch/CUDA, without installing GPU packages.
# This does NOT certify a future locked E5/reranker torch environment.
subprocess.run(
    [
        sys.executable,
        str(SOURCE / "scripts/runtime_probe.py"),
        "--require-gpu",
        "--output",
        str(WORK / "kernel-gpu-probe.json"),
    ],
    check=True,
)
# Record the actual isolated interpreter separately.
subprocess.run(
    BASE
    + [
        "python",
        str(SOURCE / "scripts/runtime_probe.py"),
        "--output",
        str(WORK / "project-runtime.json"),
    ],
    env=ENV,
    cwd=SOURCE,
    check=True,
)

In [ ]:
subprocess.run(
    BASE
    + [
        "python",
        str(SOURCE / "scripts/kaggle_day05.py"),
        "--input",
        str(INPUT),
        "--output",
        str(WORK / "output"),
    ],
    env=ENV,
    cwd=SOURCE,
    check=True,
)

In [ ]:
import zipfile

from IPython.display import FileLink, display

reports = WORK / "day05-compatibility-reports.zip"
with zipfile.ZipFile(reports, "x", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in (
        WORK / "kernel-gpu-probe.json",
        WORK / "project-runtime.json",
        INPUT / "launcher-input.json",
    ):
        archive.write(path, path.name)
for path in (WORK / "output/day05-baseline-export.zip", reports):
    print(path, "SHA256:", hashlib.sha256(path.read_bytes()).hexdigest())
    display(FileLink(str(path)))
print("Download both ZIPs, verify them locally, save outputs and stop the GPU session.")